# Experiment Pipeline

Run batch DDSP timbre transfer and WORLD vocoder baseline on the full voice dataset.

1. **DDSP inference** — `run_synthesize_dir` processes all files in `data/raw/voice/FULL/` through the trained DDSP model.
2. **Baseline inference** — `run_vocoder_dir` processes the same dataset using the WORLD vocoder with a source bank from `data/raw/solo_violin/`.

In [1]:
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

print(f"Project root: {PROJECT_ROOT}")

Project root: /m/home/home3/37/thieun1/unix/Project/final_project


In [ ]:
from evaluation.experiment_pipeline import run_synthesize_dir, run_vocoder_dir

## Pause / Resume

The pipeline automatically resumes from where it left off — already-processed files are skipped on re-run.

- **To pause**: interrupt the kernel (`Kernel > Interrupt` or `I, I` in Jupyter). Files written so far are preserved.
- **To resume**: just re-run the pipeline cell. Completed files are detected and skipped automatically.
- **`skipped`** in the output = files whose output already existed (valid WAV > 44 bytes).

## Paths

Adjust these if your directory layout differs.

In [3]:
# --- Input ---
INPUT_DIR = PROJECT_ROOT / "data" / "raw" / "voice" / "FULL"
SOURCE_DIR = PROJECT_ROOT / "data" / "raw" / "solo_violin"

# --- DDSP model ---
MODEL_DIR = PROJECT_ROOT / "artifacts" / "solo_instrument"
GIN_FILE = MODEL_DIR / "operative_config-0.gin"

# --- Output ---
DDSP_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_transfered"
BASELINE_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_baseline"

print(f"Input dir:    {INPUT_DIR}  (exists: {INPUT_DIR.exists()})")
print(f"Source dir:   {SOURCE_DIR}  (exists: {SOURCE_DIR.exists()})")
print(f"Model dir:    {MODEL_DIR}  (exists: {MODEL_DIR.exists()})")
print(f"Gin file:     {GIN_FILE}  (exists: {GIN_FILE.exists()})")
print(f"DDSP output:  {DDSP_OUTPUT_DIR}")
print(f"BL output:    {BASELINE_OUTPUT_DIR}")

Input dir:    /m/home/home3/37/thieun1/unix/Project/final_project/data/raw/voice/FULL  (exists: True)
Source dir:   /m/home/home3/37/thieun1/unix/Project/final_project/data/raw/solo_violin  (exists: True)
Model dir:    /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/solo_instrument  (exists: True)
Gin file:     /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/solo_instrument/operative_config-0.gin  (exists: True)
DDSP output:  /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_transfered
BL output:    /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_baseline


## 1. DDSP Timbre Transfer

Loads the model once, then processes all WAV files via feature-level chunking.

In [4]:
ddsp_result = run_synthesize_dir(
    model_dir=MODEL_DIR,
    gin_file=GIN_FILE,
    input_dir=INPUT_DIR,
    output_dir=DDSP_OUTPUT_DIR,
    auto_adjust=True,
    pitch_shift=0.0,
    loudness_shift=0.0,
)

print(f"\nDDSP — processed: {ddsp_result['processed']}, skipped: {ddsp_result['skipped']}, failed: {ddsp_result['failed']}")
if ddsp_result["failed_files"]:
    print("Failed files:")
    for f in ddsp_result["failed_files"]:
        print(f"  {f}")

2026-03-29 21:12:16,191 INFO experiment_pipeline: DDSP pipeline: 3095 files in /m/home/home3/37/thieun1/unix/Project/final_project/data/raw/voice/FULL
2026-03-29 21:12:16.316698: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-03-29 21:12:16.358141: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-03-29 21:12:16.360295: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-03-29 21:12:16.362567: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network L


DDSP — processed: 0, skipped: 3095, failed: 0


## 2. WORLD Vocoder Baseline

Builds a source bank from solo violin recordings, then runs F0 transfer on all target files.

In [5]:
import os

# Set n_workers > 1 to parallelise across CPU cores.
# 1 = sequential (default, safest). os.cpu_count() = use all cores.
N_WORKERS = max(1, (os.cpu_count() or 1) // 2)

baseline_result = run_vocoder_dir(
    input_dir=INPUT_DIR,
    output_dir=BASELINE_OUTPUT_DIR,
    source_dir=SOURCE_DIR,
    method="f0_ap",
    seed=42,
    n_workers=N_WORKERS,
)

print(f"\nBaseline — processed: {baseline_result['processed']}, skipped: {baseline_result['skipped']}, failed: {baseline_result['failed']}")
print(f"  (ran with {N_WORKERS} workers)")
if baseline_result["failed_files"]:
    print("Failed files:")
    for f in baseline_result["failed_files"]:
        print(f"  {f}")

2026-03-29 21:12:27,937 INFO experiment_pipeline: Baseline pipeline: 3095 files in /m/home/home3/37/thieun1/unix/Project/final_project/data/raw/voice/FULL
2026-03-29 21:12:29,083 WARNING tensorflow: Detecting that an object or model or tf.train.Checkpoint is being deleted with unrestored values. See the following logs for the specific values in question. To silence these warnings, use `status.expect_partial()`. See https://www.tensorflow.org/api_docs/python/tf/train/Checkpoint#restorefor details about the status object returned by the restore function.
2026-03-29 21:12:29,086 WARNING tensorflow: Value in checkpoint could not be found in the restored object: (root).optimizer._iterations
2026-03-29 21:12:29,087 WARNING tensorflow: Value in checkpoint could not be found in the restored object: (root).optimizer._current_learning_rate
2026-03-29 21:12:29,090 WARNING tensorflow: Value in checkpoint could not be found in the restored object: (root).optimizer._variables.1
2026-03-29 21:12:29,0


Baseline — processed: 1180, skipped: 1915, failed: 0
  (ran with 8 workers)


## Results Summary

In [6]:
print("=" * 50)
print("Experiment Pipeline Results")
print("=" * 50)
print(f"DDSP:     {ddsp_result['processed']} new / {ddsp_result['skipped']} skipped / {ddsp_result['failed']} failed")
print(f"Baseline: {baseline_result['processed']} new / {baseline_result['skipped']} skipped / {baseline_result['failed']} failed")
print(f"\nOutputs saved to:")
print(f"  DDSP:     {DDSP_OUTPUT_DIR}")
print(f"  Baseline: {BASELINE_OUTPUT_DIR}")

Experiment Pipeline Results
DDSP:     0 new / 3095 skipped / 0 failed
Baseline: 1180 new / 1915 skipped / 0 failed

Outputs saved to:
  DDSP:     /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_transfered
  Baseline: /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_baseline
